In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章把《Attention Is All You Need》里几个核心公式 / 图表复现一遍：
# 缩放注意力、sinusoidal 位置编码、复杂度表、warmup 调度、label smoothing、参数量。
# 全程在维度 <= 512 的小张量上跑，CPU 秒级完成，不需要 GPU。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# torch / matplotlib: Colab 默认已装，纯 CPU 跑小张量 + 画图，故意【不】加 -U。
# transformers:       第 7.8 节用 AutoConfig 读 Qwen3-8B config，Qwen3 要求 >=4.51。
!pip install -q -U "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 复现论文 3.2.1——为什么要除以 sqrt(d_k)
# ============================================================
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def dot_stats(d_k, n=1000):
    """造 n 对 d_k 维的随机 query/key（均值0方差1），
    返回未缩放 / 缩放后点积的方差，以及 softmax 后的最大权重（越接近1越饱和）。"""
    q = torch.randn(n, d_k)                 # [n, d_k]，每个分量 ~ N(0,1)
    k = torch.randn(n, d_k)                 # [n, d_k]
    dot = (q * k).sum(-1)                   # [n]，逐对点积 q·k = sum_i q_i k_i
    scaled = dot / (d_k ** 0.5)             # 除以 sqrt(d_k) 把方差拉回 ~1
    # 复用上面造的 key：用第一条 query 对前 8 个 key 打分，看 softmax 尖不尖
    # （和下面画图循环同一套写法 query @ key.T，别混成两个实验）
    scores = q[:1] @ k[:8].T                # [1, 8] 未缩放打分（1 条 query 对 8 个 key）
    w_raw = F.softmax(scores, dim=-1).max().item()
    w_scaled = F.softmax(scores / d_k ** 0.5, dim=-1).max().item()
    return dot.var().item(), scaled.var().item(), w_raw, w_scaled

print(f"{'d_k':>5} | {'var(未缩放)':>12} | {'var(缩放后)':>12} | {'softmax max(未缩放)':>20} | {'softmax max(缩放)':>18}")
for d_k in [8, 64, 512]:
    v_raw, v_scaled, w_raw, w_scaled = dot_stats(d_k)
    print(f"{d_k:>5} | {v_raw:>12.2f} | {v_scaled:>12.2f} | {w_raw:>20.3f} | {w_scaled:>18.3f}")

# 结论：未缩放点积的方差随 d_k 线性增长（≈ d_k），softmax 越来越尖（max 权重趋近 1，
# 意味着几乎把全部注意力集中到一个位置上、梯度趋近 0）；除以 sqrt(d_k) 后方差稳定在 ~1，
# softmax 保持温和——这正是论文加这个缩放因子的原因。

# 把"缩放前后 softmax 有多尖"画成图：遍历一组 d_k，每个点取 200 次平均
import matplotlib.pyplot as plt
dks = [4, 8, 16, 32, 64, 128, 256, 512]
raw_w, scaled_w = [], []
for dk in dks:
    r, s = [], []
    for _ in range(200):
        qq = torch.randn(1, dk); kk = torch.randn(8, dk)     # 1 条 query 对 8 个 key
        sc = qq @ kk.T                                         # [1, 8] 打分
        r.append(F.softmax(sc, -1).max().item())              # 未缩放的 softmax 最大权重
        s.append(F.softmax(sc / dk ** 0.5, -1).max().item())  # 缩放后的
    raw_w.append(sum(r) / len(r)); scaled_w.append(sum(s) / len(s))

plt.figure(figsize=(7, 4))
plt.plot(dks, raw_w, "o-", label="unscaled")
plt.plot(dks, scaled_w, "s-", label="scaled by sqrt(d_k)")
plt.xscale("log", base=2); plt.ylim(0, 1)
plt.xlabel("d_k"); plt.ylabel("avg softmax max weight")
plt.title("Scaling by sqrt(d_k) keeps softmax from saturating")
plt.legend(); plt.tight_layout(); plt.show()
# 观察：未缩放时 softmax 最大权重随 d_k 升高、逼近 1（饱和 -> 梯度趋 0）；
# 缩放后基本平在低位、不随 d_k 恶化——正是论文加 sqrt(d_k) 的效果。

In [ ]:
# ============================================================
# Cell 3: 复现论文 3.5——sinusoidal 位置编码
# ============================================================
import torch
import matplotlib.pyplot as plt

def sinusoidal_pe(max_len, d_model):
    """按论文公式造 [max_len, d_model] 的位置编码矩阵：
    PE[pos, 2i]   = sin(pos / 10000^(2i/d_model))
    PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))"""
    pos = torch.arange(max_len).unsqueeze(1)                       # [max_len, 1]
    i = torch.arange(0, d_model, 2)                                # [d_model/2]，偶数维下标 0,2,4,...（就是公式里的 2i，故下面 i/d_model 即 2i/d_model）
    div = torch.pow(10000, i / d_model)                            # [d_model/2]，各维的"波长"分母
    pe = torch.zeros(max_len, d_model)                             # [max_len, d_model]
    pe[:, 0::2] = torch.sin(pos / div)                             # 偶数维放 sin
    pe[:, 1::2] = torch.cos(pos / div)                             # 奇数维放 cos
    return pe

pe = sinusoidal_pe(max_len=64, d_model=128)
print("PE 形状:", tuple(pe.shape))
print("每个位置的向量都不同吗?", (pe.unique(dim=0).shape[0] == pe.shape[0]))

plt.figure(figsize=(9, 4))
plt.imshow(pe.T, aspect="auto", cmap="RdBu")                       # 转置成 [d_model, max_len]，行=维度、列=位置
plt.xlabel("position")
plt.ylabel("dimension")
plt.title("Sinusoidal Positional Encoding (max_len=64, d_model=128)")
plt.colorbar(label="value")
plt.tight_layout()
plt.show()
# 观察：低维（图上方）波长短、随位置快速振荡；高维（图下方）波长长、变化缓慢——
# 每个位置由这一整列不同频率的采样值唯一确定，像一串"多进制位置指纹"。

In [ ]:
# ============================================================
# Cell 4: 复现论文第 4 节——自注意力 vs RNN 的复杂度 / 并行性对比
# ============================================================
import torch, time

d = 512                                          # 表示维度 d_model
def self_attn_once(x):
    """一次并行自注意力：串行操作 O(1)（就是几个矩阵乘）。x: [n, d] -> [n, d]。"""
    scores = x @ x.T / d ** 0.5                   # [n, n]，一次算完所有位置对的打分
    w = torch.softmax(scores, dim=-1)             # [n, n]
    return w @ x                                  # [n, d]

def rnn_scan(x, Whh):
    """RNN 式串行扫描：串行操作 O(n)，第 t 步必须等第 t-1 步。x: [n, d] -> [n, d]。"""
    h = torch.zeros(d)
    outs = []
    for t in range(x.shape[0]):                   # 这个 for 循环没法并行——正是 RNN 的顺序瓶颈
        h = torch.tanh(x[t] + h @ Whh)            # h_t 依赖 h_{t-1}
        outs.append(h)
    return torch.stack(outs)

torch.manual_seed(0)
Whh = torch.randn(d, d) * 0.01
print(f"{'n':>6} | {'self-attn (ms)':>16} | {'rnn-scan (ms)':>16} | {'加速比':>8}")
for n in [16, 64, 256, 1024]:
    x = torch.randn(n, d)
    t0 = time.perf_counter(); _ = self_attn_once(x); t_attn = (time.perf_counter() - t0) * 1e3
    t0 = time.perf_counter(); _ = rnn_scan(x, Whh); t_rnn = (time.perf_counter() - t0) * 1e3
    print(f"{n:>6} | {t_attn:>16.2f} | {t_rnn:>16.2f} | {t_rnn / max(t_attn, 1e-6):>7.1f}x")

# 观察：RNN 的耗时随 n 近似线性增长（那个 for 循环 O(n) 串行步没法并行）；
# 自注意力一次矩阵乘就并行算完（串行 O(1)），小 n 下明显更快。但加速比随 n 收窄。

In [ ]:
# ============================================================
# Cell 5: 复现论文 5.3——Adam 的 warmup 学习率调度
# ============================================================
import matplotlib.pyplot as plt

def transformer_lr(step, d_model=512, warmup=4000):
    """论文公式: lrate = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))"""
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

steps = list(range(1, 20000))
lrs = [transformer_lr(s) for s in steps]
peak_step = max(range(len(lrs)), key=lambda i: lrs[i]) + 1
print(f"峰值出现在第 {peak_step} 步（应等于 warmup=4000），峰值 lr ≈ {max(lrs):.2e}")

plt.figure(figsize=(8, 4))
plt.plot(steps, lrs)
plt.axvline(4000, color="red", ls="--", label="warmup=4000")
plt.xlabel("training step")
plt.ylabel("learning rate")
plt.title("Transformer LR Schedule: warmup + inverse-sqrt decay (d_model=512)")
plt.legend()
plt.tight_layout()
plt.show()
# 观察：前 4000 步 lr 从 0 线性升到峰值（min 取第二项 step*warmup^-1.5，正比于 step）；
# 4000 步后按 1/sqrt(step) 缓降（min 取第一项）。峰值恰好落在 warmup=4000。

In [ ]:
# ============================================================
# Cell 6: 复现论文 5.4——label smoothing 把 one-hot 目标抹平
# ============================================================
import torch
import torch.nn.functional as F

vocab = 6                                        # 假设词表只有 6 个 token，正确答案是第 2 个（下标 2）
target = 2
eps = 0.1                                         # 论文的 label smoothing 系数 eps_ls = 0.1

one_hot = torch.zeros(vocab); one_hot[target] = 1.0                 # 原始硬标签：正确项 1、其余 0
smooth = torch.full((vocab,), eps / (vocab - 1)); smooth[target] = 1 - eps  # 软标签：正确 0.9、其余均分 0.1
print("硬标签 (one-hot):", one_hot.tolist())
print("软标签 (smoothed):", [round(x, 3) for x in smooth.tolist()])

# 用一个"还算自信"的模型输出，看两种目标下的交叉熵损失
logits = torch.tensor([0.1, 0.2, 3.0, 0.1, 0.4, 0.2])              # 模型给正确项(2)较高 logit
logp = F.log_softmax(logits, dim=-1)
loss_hard = -(one_hot * logp).sum()                                # 只惩罚正确项的 -log p
loss_soft = -(smooth * logp).sum()                                 # 软标签下的交叉熵
print(f"\n硬标签交叉熵: {loss_hard.item():.4f}")
print(f"软标签交叉熵: {loss_soft.item():.4f}")
print("软标签的 loss 更大：主要来自软目标本身带熵、把交叉熵下界从 0 抬高了，"
      "\n其次来自它更重地惩罚'把正确项概率推向极端'的过度自信。"
      "\n注意区分：论文说的 perplexity 变差，指用标签平滑【训练出的模型】在标准困惑度上更差，"
      "\n不是这里同一模型换目标算出的 loss；但两者都指向'别太笃定'，换来更好的泛化 / BLEU。")

# 把硬标签 vs 软标签画成并排柱状图
import matplotlib.pyplot as plt
import numpy as np
x = np.arange(vocab)
plt.figure(figsize=(7, 4))
bw = 0.38
plt.bar(x - bw / 2, one_hot.numpy(), width=bw, label="hard (one-hot)")
plt.bar(x + bw / 2, smooth.numpy(), width=bw, label="smoothed (eps=0.1)")
plt.xlabel("token id"); plt.ylabel("target probability")
plt.title("Label Smoothing: one-hot vs smoothed target (correct id=2)")
plt.legend(); plt.tight_layout(); plt.show()
# 观察：正确项(id=2)从 1.0 降到 0.9，匀出的 0.1 平摊给其余 5 项（各 0.02）。
# 说明：这是"把 eps 只分给其余 token"的常见变体（正确项恰好 = 1-eps = 0.9）；
# 论文原版是把 eps 按均匀分布撒到全部词表（含正确项），两者都是合法的 label smoothing。

In [ ]:
# ============================================================
# Cell 7: 按论文超参数清 base / big 模型的参数量
# ============================================================
def transformer_params(N, d_model, d_ff, vocab=37000):
    """粗算原版 Transformer（encoder-decoder）参数量。返回百万(M)为单位。
    只算主要权重（忽略 LayerNorm 的 gamma/beta、bias 这些小头），量级足够对上论文。
    位置编码是 sinusoidal、无可学习参数，故不计入。"""
    # 每个注意力：4 个 d_model×d_model 投影（W_Q, W_K, W_V, W_O）
    attn = 4 * d_model * d_model
    # 每个 FFN：d_model×d_ff 升维 + d_ff×d_model 降维
    ffn = 2 * d_model * d_ff
    # 编码器每层 = 1 个自注意力 + 1 个 FFN
    enc_layer = attn + ffn
    # 解码器每层 = 自注意力 + cross-attention + FFN
    dec_layer = 2 * attn + ffn
    blocks = N * enc_layer + N * dec_layer
    # embedding（源+目标+输出，weight tying 下共享，这里算一份 d_model×vocab）
    embed = vocab * d_model
    total = blocks + embed
    return total / 1e6

base = transformer_params(N=6, d_model=512, d_ff=2048)
big = transformer_params(N=6, d_model=1024, d_ff=4096)
print(f"base 模型 (d_model=512,  d_ff=2048, h=8):  {base:6.1f} M 参数（论文 ~65M）")
print(f"big  模型 (d_model=1024, d_ff=4096, h=16): {big:6.1f} M 参数（论文 ~213M）")
print("\n量级对得上——多头的头数 h 不改变参数量（4 个投影矩阵与 h 无关，第 7 章算过），"
      "\n所以 base/big 的差别主要来自 d_model、d_ff 翻倍带来的平方级增长。")

In [ ]:
# ============================================================
# Cell 8: 读 Qwen3-8B config，对照"2017 原版 vs 今天"
# ============================================================
# AutoConfig 只下载几 KB 的 config.json，不下载几个 GB 的权重，CPU 秒级完成。
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")

# rope_theta（RoPE 的 base θ）在 config 里的位置随 transformers 版本而变：4.x 在顶层
# cfg.rope_theta；5.x 起挪进嵌套字典 cfg.rope_parameters["rope_theta"]。两处都试，兼容新旧版本。
rope_theta = getattr(cfg, "rope_theta", None)
if rope_theta is None:
    rope_theta = (getattr(cfg, "rope_parameters", None) or {}).get("rope_theta")

print("Qwen3-8B 关键配置：")
print(f"  层数         num_hidden_layers   = {cfg.num_hidden_layers}   (原版 N=6)")
print(f"  隐藏维度     hidden_size         = {cfg.hidden_size}  (原版 d_model=512)")
print(f"  FFN 中间维度 intermediate_size   = {cfg.intermediate_size}  (原版 d_ff=2048)")
print(f"  query 头数   num_attention_heads = {cfg.num_attention_heads}   (原版 h=8)")
print(f"  KV 头数      num_key_value_heads = {cfg.num_key_value_heads}    (原版=query 头数, 即 MHA)")
print(f"  RoPE base    rope_theta          = {rope_theta}  (原版用 sinusoidal, 无此参数)")
print(f"  归一化 eps   rms_norm_eps        = {cfg.rms_norm_eps}  (RMSNorm; 原版是 LayerNorm)")
print(f"  激活函数     hidden_act          = {cfg.hidden_act}  (SwiGLU 门控; 原版是 ReLU)")
print("\n对照：query 头数 > KV 头数 => GQA（原版 MHA 两者相等）；有 rope_theta => RoPE"
      "（原版 sinusoidal）；hidden_act=silu + gated => SwiGLU（原版 ReLU）。同一套 2017 年的骨架，"
      "\n零件逐个换成了更好的版本——这正是第 12 章要逐项展开的现代改动。")